# XGBoost Baseline — Sinhala Hate Speech Detection on Noisy (Transliterated) Text

This notebook implements the **traditional ML baseline (Section 5.3.1)** of the project proposal,
using **XGBoost**, trained and tested on the **noisy transliteration data** (`noisy_text` field).



**What this notebook does, matching the proposal exactly:**
- Trains and evaluates separately on **light** and **heavy** transliteration severity (Section 5.1.2).
- Uses only the feature representation Section 5.3.1 specifies: **character n-gram (1-5) TF-IDF**
  and **word n-gram (1-2) TF-IDF**.
- Handles **class imbalance** (proposal notes ~41% offensive) via `scale_pos_weight`.
- Tunes hyperparameters with **leakage-safe** cross-validation (TF-IDF refit per fold).
- Reports **macro-F1**, per-class precision/recall/F1, weighted F1, confusion matrix, and compares
  against a majority-class baseline (Section 5.4.1).
- Saves the trained pipeline (vectorizers + model together) and result tables back to Google Drive.

**Colab free-tier friendly:** refitting TF-IDF inside every CV fold is more expensive than fitting
it once, so this notebook uses a **disk cache** (`Pipeline(memory=...)`) so that repeated
hyperparameter candidates which only change the XGBoost settings (not the TF-IDF settings) reuse
the already-computed TF-IDF features for that fold instead of recomputing them. GPU is
auto-detected and used if available, but not required.


## 1. Setup — install packages

In [1]:
!pip install -q --upgrade xgboost scikit-learn scipy joblib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 6.0 MB/s eta 0:00:00


## 2. Mount Google Drive and set paths
Edit `BASE_DIR` below to point at your `transliterations` folder.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

# >>> EDIT THIS to the path of your "transliterations" folder in Drive <<<
BASE_DIR = "/content/drive/MyDrive/transliterations"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")
RESULTS_DIR = os.path.join(BASE_DIR, "results", "xgboost")
os.makedirs(RESULTS_DIR, exist_ok=True)

SEVERITIES = ["light", "heavy"]   # transliteration_light.jsonl / transliteration_heavy.jsonl

TRAIN_FILES = {sev: os.path.join(TRAIN_DIR, f"transliteration_{sev}.jsonl") for sev in SEVERITIES}
TEST_FILES  = {sev: os.path.join(TEST_DIR,  f"transliteration_{sev}.jsonl") for sev in SEVERITIES}

for sev in SEVERITIES:
    print(sev, "train:", TRAIN_FILES[sev], os.path.exists(TRAIN_FILES[sev]))
    print(sev, "test :", TEST_FILES[sev],  os.path.exists(TEST_FILES[sev]))


Mounted at /content/drive
light train: /content/drive/MyDrive/transliterations/train/transliteration_light.jsonl True
light test : /content/drive/MyDrive/transliterations/test/transliteration_light.jsonl True
heavy train: /content/drive/MyDrive/transliterations/train/transliteration_heavy.jsonl True
heavy test : /content/drive/MyDrive/transliterations/test/transliteration_heavy.jsonl True


## 3. Imports

In [3]:
import json
import shutil
import tempfile
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report, f1_score, precision_recall_fscore_support,
    confusion_matrix, accuracy_score
)
from scipy.stats import randint, uniform

import xgboost as xgb

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Detect GPU for XGBoost (optional -- code works fine on CPU-only Colab too)
try:
    import subprocess
    _gpu_ok = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    _gpu_ok = False

XGB_DEVICE = "cuda" if _gpu_ok else "cpu"
XGB_TREE_METHOD = "hist"
print("XGBoost device:", XGB_DEVICE)


XGBoost device: cuda


## 4. Load the noisy JSONL data

We train and test using the **`noisy_text`** field (not `original_text`) — the model only ever sees noisy transliterated text.

In [4]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

data = {}
for sev in SEVERITIES:
    data[sev] = {
        "train": load_jsonl(TRAIN_FILES[sev]),
        "test":  load_jsonl(TEST_FILES[sev]),
    }
    for split in ("train", "test"):
        df = data[sev][split]
        assert "noisy_text" in df.columns and "label" in df.columns, \
            f"Missing expected columns in {sev}/{split}"
        df["noisy_text"] = df["noisy_text"].astype(str)
        df["label"] = df["label"].astype(int)
    print(f"[{sev}] train: {len(data[sev]['train'])} rows | test: {len(data[sev]['test'])} rows")
    print(f"[{sev}] train label distribution:\n{data[sev]['train']['label'].value_counts(normalize=True)}")


[light] train: 7500 rows | test: 2500 rows
[light] train label distribution:
label
0    0.576533
1    0.423467
Name: proportion, dtype: float64
[heavy] train: 7500 rows | test: 2500 rows
[heavy] train label distribution:
label
0    0.576533
1    0.423467
Name: proportion, dtype: float64


## 5. Feature engineering

Per Section 5.3.1: **character n-gram (1-5) TF-IDF** + **word n-gram (1-2) TF-IDF**, nothing else.

The key change: instead of calling `.fit_transform()` on the whole training set up front, both
TF-IDF vectorizers are wrapped inside a `FeatureUnion`, which is itself the first step of a
`Pipeline` whose last step is the XGBoost classifier. When this pipeline is handed to
`RandomizedSearchCV` with **raw text** (not a pre-built matrix), scikit-learn clones and refits
the *entire* pipeline — including both TF-IDF vectorizers — separately on each CV fold's training
portion. The held-out fold's text never touches vectorizer fitting for that fold. This removes the
TF-IDF leakage.

A `memory` cache is attached to the pipeline so that when `RandomizedSearchCV` tries different
XGBoost hyperparameters on the *same* fold, it doesn't needlessly recompute the (unchanged) TF-IDF
features every single time — only re-run the classifier step. This keeps things fast enough for
Colab's free tier despite the extra refitting.


In [5]:
_cache_dir = tempfile.mkdtemp(prefix="tfidf_cache_")

def build_pipeline(scale_pos_weight, cache_dir=_cache_dir):
    char_vec = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(1, 5),
        min_df=2,
        max_features=30000,
        sublinear_tf=True,
    )
    word_vec = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=15000,
        sublinear_tf=True,
        token_pattern=r"(?u)\b\w+\b",
    )
    features = FeatureUnion([
        ("char_tfidf", char_vec),
        ("word_tfidf", word_vec),
    ])

    clf = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method=XGB_TREE_METHOD,
        device=XGB_DEVICE,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    )

    pipe = Pipeline(
        [("features", features), ("clf", clf)],
        memory=cache_dir,  # caches the (expensive, fold-specific) TF-IDF output
    )
    return pipe


## 6. Hyperparameter tuning + training (per severity)

`RandomizedSearchCV` now receives **raw noisy text** (`train_df["noisy_text"].values`), not a
pre-built feature matrix. It internally splits the raw text into 5 stratified folds, and for each
fold, clones and fits the *whole pipeline* (TF-IDF + XGBoost) using only that fold's training
portion, then scores it on the held-out portion. `scale_pos_weight` is computed once from the full
training label distribution (this only uses labels, not text/features, so it is not a source of
feature leakage).


In [6]:
def train_xgb_for_severity(sev, n_iter=20, cv_folds=5):
    print(f"\n{'='*70}\nTraining XGBoost on '{sev}' transliteration noise\n{'='*70}")

    train_df = data[sev]["train"].reset_index(drop=True)
    test_df  = data[sev]["test"].reset_index(drop=True)

    X_train_text = train_df["noisy_text"].values
    y_train = train_df["label"].values
    X_test_text = test_df["noisy_text"].values
    y_test  = test_df["label"].values

    n_pos = (y_train == 1).sum()
    n_neg = (y_train == 0).sum()
    spw = n_neg / max(n_pos, 1)
    print(f"Class balance -> not-offensive: {n_neg}, offensive: {n_pos}, scale_pos_weight={spw:.3f}")

    pipe = build_pipeline(scale_pos_weight=spw)

    param_dist = {
        "clf__n_estimators": randint(150, 600),
        "clf__max_depth": randint(3, 10),
        "clf__learning_rate": uniform(0.01, 0.29),
        "clf__subsample": uniform(0.6, 0.4),
        "clf__colsample_bytree": uniform(0.5, 0.5),
        "clf__min_child_weight": randint(1, 8),
        "clf__gamma": uniform(0, 0.5),
        "clf__reg_alpha": uniform(0, 1.0),
        "clf__reg_lambda": uniform(0.5, 2.0),
    }

    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        pipe,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="f1_macro",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
    )
    # NOTE: raw text goes in here, not a pre-built TF-IDF matrix.
    # RandomizedSearchCV + Pipeline handles fold-wise TF-IDF fitting internally.
    search.fit(X_train_text, y_train)

    print("Best CV macro-F1:", round(search.best_score_, 4))
    print("Best params:", search.best_params_)

    best_pipeline = search.best_estimator_  # already refit on the FULL training set by RandomizedSearchCV (refit=True default)

    artifacts = {
        "pipeline": best_pipeline,
        "best_params": search.best_params_,
        "cv_macro_f1": search.best_score_,
    }
    return artifacts, X_test_text, y_test, test_df


## 7. Evaluation (matches Section 5.4.1 / 5.4.2)

Same as before: macro-F1, per-class precision/recall/F1, weighted F1, confusion matrix, and a
majority-class baseline comparison. The fitted pipeline handles the raw-text -> TF-IDF ->
prediction steps in one call, using the vectorizers it fit on the full training set (this final
refit-on-everything step, done once after hyperparameter selection, is standard and not leakage —
leakage only concerns what happens *during* cross-validation/model selection).


In [7]:
def evaluate(artifacts, X_test_text, y_test, test_df, sev):
    pipeline = artifacts["pipeline"]
    y_pred = pipeline.predict(X_test_text)

    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, support = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1])

    print(f"\n--- Results: severity = {sev} ---")
    print(f"Accuracy     : {acc:.4f}")
    print(f"Macro-F1     : {macro_f1:.4f}")
    print(f"Weighted-F1  : {weighted_f1:.4f}")
    print(classification_report(y_test, y_pred, target_names=["not-offensive", "offensive"], digits=4))

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    print("Confusion matrix [rows=true, cols=pred] (0=not-offensive, 1=offensive):")
    print(cm)

    dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
    dummy.fit(X_test_text.reshape(-1, 1), y_test)
    y_dummy = dummy.predict(X_test_text.reshape(-1, 1))
    dummy_f1 = f1_score(y_test, y_dummy, average="macro")
    print(f"Majority-class baseline macro-F1: {dummy_f1:.4f}")

    results_row = {
        "severity": sev,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "precision_not_offensive": prec[0], "recall_not_offensive": rec[0], "f1_not_offensive": f1[0],
        "precision_offensive": prec[1], "recall_offensive": rec[1], "f1_offensive": f1[1],
        "support_not_offensive": int(support[0]), "support_offensive": int(support[1]),
        "majority_baseline_macro_f1": dummy_f1,
        "cv_macro_f1": artifacts["cv_macro_f1"],
        "best_params": artifacts["best_params"],
    }

    preds_df = test_df.copy()
    preds_df["predicted_label"] = y_pred
    preds_df["predicted_proba_offensive"] = pipeline.predict_proba(X_test_text)[:, 1]

    return results_row, preds_df, cm


## 8. Run the full pipeline for both severities and save everything to Drive

In [8]:
all_results = []
all_cms = {}
all_artifacts = {}

for sev in SEVERITIES:
    artifacts, X_test_text, y_test, test_df = train_xgb_for_severity(sev, n_iter=20, cv_folds=5)
    results_row, preds_df, cm = evaluate(artifacts, X_test_text, y_test, test_df, sev)
    all_results.append(results_row)
    all_cms[sev] = cm
    all_artifacts[sev] = artifacts

    # --- save the whole pipeline (vectorizers + model together) for this severity ---
    model_path = os.path.join(RESULTS_DIR, f"xgboost_pipeline_{sev}.joblib")
    joblib.dump(artifacts["pipeline"], model_path)
    print(f"Saved pipeline -> {model_path}")

    # --- save test predictions ---
    preds_path = os.path.join(RESULTS_DIR, f"test_predictions_{sev}.csv")
    preds_df.to_csv(preds_path, index=False)
    print(f"Saved predictions -> {preds_path}")

# --- save summary results table ---
results_df = pd.DataFrame(all_results)
results_path = os.path.join(RESULTS_DIR, "xgboost_results_summary.csv")
results_df.to_csv(results_path, index=False)
print(f"\nSaved results summary -> {results_path}")
results_df



Training XGBoost on 'light' transliteration noise
Class balance -> not-offensive: 4324, offensive: 3176, scale_pos_weight=1.361
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best CV macro-F1: 0.8057
Best params: {'clf__colsample_bytree': np.float64(0.8047821669899484), 'clf__gamma': np.float64(0.25133951161443074), 'clf__learning_rate': np.float64(0.02492883786249691), 'clf__max_depth': 7, 'clf__min_child_weight': 1, 'clf__n_estimators': 534, 'clf__reg_alpha': np.float64(0.23956189066697242), 'clf__reg_lambda': np.float64(0.7897897441824462), 'clf__subsample': np.float64(0.7957811041110252)}

--- Results: severity = light ---
Accuracy     : 0.8232
Macro-F1     : 0.8128
Weighted-F1  : 0.8211
               precision    recall  f1-score   support

not-offensive     0.8249    0.8916    0.8570      1485
    offensive     0.8201    0.7232    0.7686      1015

     accuracy                         0.8232      2500
    macro avg     0.8225    0.8074    0.8128      2500
 weigh

,severity,accuracy,macro_f1,weighted_f1,precision_not_offensive,recall_not_offensive,f1_not_offensive,precision_offensive,recall_offensive,f1_offensive,support_not_offensive,support_offensive,majority_baseline_macro_f1,cv_macro_f1,best_params
0,light,0.8232,0.812772,0.821079,0.824922,0.891582,0.856958,0.820112,0.723153,0.768586,1485,1015,0.372647,0.805699,"{'clf__colsample_bytree': 0.8047821669899484, ..."
1,heavy,0.8152,0.804143,0.812892,0.817899,0.886195,0.850679,0.810325,0.711330,0.757608,1485,1015,0.372647,0.803041,"{'clf__colsample_bytree': 0.8047821669899484, ..."


## 9. Degradation check: does heavy noise hurt more than light noise?

In [9]:
if set(SEVERITIES) == {"light", "heavy"}:
    f1_light = results_df.loc[results_df.severity == "light", "macro_f1"].values[0]
    f1_heavy = results_df.loc[results_df.severity == "heavy", "macro_f1"].values[0]
    drop = f1_light - f1_heavy
    print(f"Macro-F1 (light noise): {f1_light:.4f}")
    print(f"Macro-F1 (heavy noise): {f1_heavy:.4f}")
    print(f"Performance drop (light -> heavy): {drop:.4f}  "
          f"({'heavier noise hurts more' if drop > 0 else 'no clear extra degradation from heavier noise'})")


Macro-F1 (light noise): 0.8128
Macro-F1 (heavy noise): 0.8041
Performance drop (light -> heavy): 0.0086  (heavier noise hurts more)


## 10. (Optional) Feature importance — which signals matter most?

In [10]:
import matplotlib.pyplot as plt

def show_top_features(sev, top_n=20):
    pipeline = all_artifacts[sev]["pipeline"]
    feature_union = pipeline.named_steps["features"]
    model = pipeline.named_steps["clf"]

    all_names = feature_union.get_feature_names_out()
    importances = model.feature_importances_
    order = np.argsort(importances)[::-1][:top_n]

    top_feats = pd.DataFrame({
        "feature": all_names[order],
        "importance": importances[order],
    })
    print(f"Top {top_n} features for severity='{sev}':")
    display(top_feats)

    plt.figure(figsize=(8, 6))
    plt.barh(top_feats["feature"][::-1], top_feats["importance"][::-1])
    plt.title(f"Top {top_n} XGBoost feature importances ({sev} noise)")
    plt.xlabel("importance")
    plt.tight_layout()
    plt.show()

# Example:
# show_top_features("heavy")
# show_top_features("light")


## 11. Clean up the TF-IDF disk cache

In [11]:
shutil.rmtree(_cache_dir, ignore_errors=True)
print("Removed TF-IDF cache directory:", _cache_dir)


Removed TF-IDF cache directory: /tmp/tfidf_cache_p_b_4cwu
